In [ ]:
import json
import itertools
import math
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore")

from itertools import combinations, permutations
from collections import defaultdict, Counter

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


In [ ]:
# -- Utility Functions --

def euclidean_distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])


def build_distance_matrix(nodes):
    n = len(nodes)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = euclidean_distance(nodes[i], nodes[j])
            D[i, j] = d
            D[j, i] = d
    return D


def route_distance(route, D):
    return float(sum(D[route[k], route[k + 1]] for k in range(len(route) - 1)))


def infer_num_vars_from_qubo(Q):
    if not Q:
        return 0
    return 1 + max(max(i, j) for i, j in Q.keys())


def bitstring_to_array(bitstring, n):
    bits = np.array([int(b) for b in bitstring[::-1]], dtype=int)
    if len(bits) < n:
        bits = np.pad(bits, (0, n - len(bits)))
    return bits[:n]


def counts_to_marginals(counts, n):
    probs = np.zeros(n)
    total = sum(counts.values())
    for bitstring, cnt in counts.items():
        probs += cnt * bitstring_to_array(bitstring, n)
    return probs / max(1, total)


def energy_of_bitstring(bitstring_or_bits, Q, constant=0.0):
    if isinstance(bitstring_or_bits, str):
        n = infer_num_vars_from_qubo(Q)
        x = bitstring_to_array(bitstring_or_bits, n)
    else:
        x = np.asarray(bitstring_or_bits, dtype=int)

    e = constant
    for (i, j), q in Q.items():
        if i == j:
            e += q * x[i]
        else:
            e += q * x[i] * x[j]
    return float(e)


def polar_radius(depot, p):
    return euclidean_distance(depot, p)


def polar_angle(depot, p):
    return math.degrees(math.atan2(p[1] - depot[1], p[0] - depot[0])) % 360.0


def polar_radius_angle(depot, p):
    return polar_radius(depot, p), polar_angle(depot, p)


def build_polar_features(nodes):
    depot = nodes[0]
    radii = {}
    angles = {}
    for idx in range(1, len(nodes)):
        radius, angle = polar_radius_angle(depot, nodes[idx])
        radii[idx] = radius
        angles[idx] = angle
    return radii, angles


def nearest_neighbor_lists(D, customers, k):
    nearest = {}
    for customer in customers:
        ranked = sorted(
            (other for other in customers if other != customer),
            key=lambda other: (D[customer, other], other),
        )
        nearest[customer] = ranked[: min(k, len(ranked))]
    return nearest


def get_k_nearest(nodes, k):
    customers = list(range(1, len(nodes)))
    D = build_distance_matrix(nodes)
    return nearest_neighbor_lists(D, customers, k)


def circular_angular_difference(angle_a, angle_b):
    diff = abs((angle_a - angle_b) % 360.0)
    if diff > 180.0:
        diff = 360.0 - diff
    return float(diff)


def minimal_angular_span(angles):
    angles = [angle % 360.0 for angle in angles]
    if len(angles) <= 1:
        return 0.0
    ordered = sorted(angles)
    gaps = []
    for idx in range(len(ordered)):
        nxt = ordered[(idx + 1) % len(ordered)]
        gaps.append((nxt - ordered[idx]) % 360.0)
    return float(360.0 - max(gaps))


def canonical_subset(subset):
    return tuple(sorted(int(node) for node in subset))


def dedupe_canonical_subsets(subsets):
    deduped = []
    seen = set()
    for subset in subsets:
        key = canonical_subset(subset)
        if key not in seen:
            seen.add(key)
            deduped.append(key)
    return deduped


def max_pairwise_distance(subset, D):
    subset = canonical_subset(subset)
    if len(subset) <= 1:
        return 0.0
    return float(max(D[i, j] for i, j in combinations(subset, 2)))


def radial_spread(subset, radii):
    subset = canonical_subset(subset)
    if len(subset) <= 1:
        return 0.0
    subset_radii = [radii[node] for node in subset]
    return float(max(subset_radii) - min(subset_radii))


def subset_angular_span(subset, angles):
    subset = canonical_subset(subset)
    if len(subset) <= 1:
        return 0.0
    return minimal_angular_span([angles[node] for node in subset])


def subset_geometry_metrics(subset, D, radii, angles):
    subset = canonical_subset(subset)
    return {
        "max_pairwise": max_pairwise_distance(subset, D),
        "radial_spread": radial_spread(subset, radii),
        "angular_span": subset_angular_span(subset, angles),
    }


def subset_geometry(subset, nodes, D=None, radii=None, angles=None):
    if D is None:
        D = build_distance_matrix(nodes)
    if radii is None or angles is None:
        radii, angles = build_polar_features(nodes)
    metrics = subset_geometry_metrics(subset, D, radii, angles)
    return metrics["max_pairwise"], metrics["radial_spread"], metrics["angular_span"]


def compactness_density_score(
    cost,
    savings,
    angular_span,
    max_pairwise,
    radial_spread,
    dist_threshold,
    radial_threshold,
):
    geom_scale = 1.0
    geom_scale += max_pairwise / max(dist_threshold, 1e-9)
    geom_scale += radial_spread / max(radial_threshold, 1e-9)
    geom_scale += angular_span / 180.0
    return float(savings / max(cost * geom_scale, 1e-9))


In [ ]:
def exact_subset_route_cost(subset, D):
    subset = canonical_subset(subset)
    if len(subset) == 0:
        return (), 0.0
    if len(subset) == 1:
        return subset, float(2.0 * D[0, subset[0]])

    best_order, best_cost = None, float("inf")
    for perm in permutations(subset):
        cost = D[0, perm[0]]
        for k in range(len(perm) - 1):
            cost += D[perm[k], perm[k + 1]]
        cost += D[perm[-1], 0]
        if cost < best_cost:
            best_cost = cost
            best_order = perm
    return best_order, float(best_cost)


def count_candidate_subsets(n_customers, allowed_sizes):
    total = 0
    for size in sorted(set(allowed_sizes)):
        if 1 <= size <= n_customers:
            total += math.comb(n_customers, size)
    return total


def count_routes_by_size(route_pool):
    counts = Counter(route["size"] for route in route_pool)
    return {size: counts[size] for size in sorted(counts)}


def build_route_generation_context(nodes, config=None):
    config = dict(config or {})
    customers = list(range(1, len(nodes)))
    D = build_distance_matrix(nodes)
    radii, angles = build_polar_features(nodes)

    if customers:
        k_nearest = max(1, min(config.get("k_nearest", 6), max(1, len(customers) - 1)))
        nearest = nearest_neighbor_lists(D, customers, k_nearest)
        nn_distances = [D[c, nearest[c][0]] for c in customers if nearest.get(c)]
        median_nn = float(np.median(nn_distances)) if nn_distances else 1.0
    else:
        nearest = {}
        median_nn = 1.0

    return {
        "customers": customers,
        "D": D,
        "radii": radii,
        "angles": angles,
        "nearest": nearest,
        "singleton_cost": {c: float(2.0 * D[0, c]) for c in customers},
        "median_nearest_neighbor_distance": median_nn,
        "dist_threshold": float(config.get("beta_dist", 3.5) * median_nn),
        "radial_threshold": float(config.get("radial_spread_threshold", 2.25) * median_nn),
    }


def subset_passes_geometric_pruning(subset, context, config=None):
    config = dict(config or {})
    metrics = subset_geometry_metrics(
        subset,
        context["D"],
        context["radii"],
        context["angles"],
    )

    if len(subset) <= 1 or not config.get("use_geometric_pruning", True):
        return True, metrics

    angle_thresholds = config.get(
        "angle_threshold_by_size",
        {2: 180.0, 3: 130.0, 4: 95.0, 5: 80.0},
    )

    if metrics["max_pairwise"] > context["dist_threshold"]:
        return False, metrics
    if metrics["radial_spread"] > context["radial_threshold"]:
        return False, metrics
    if metrics["angular_span"] > angle_thresholds.get(len(subset), 360.0):
        return False, metrics
    return True, metrics


def score_route_entry(subset, order, cost, metrics, context, config=None):
    config = dict(config or {})
    weights = dict(config.get("score_weights", {}))

    singleton_total = sum(context["singleton_cost"][customer] for customer in subset)
    savings = float(singleton_total - cost)

    density_score = compactness_density_score(
        cost=cost,
        savings=savings,
        angular_span=metrics["angular_span"],
        max_pairwise=metrics["max_pairwise"],
        radial_spread=metrics["radial_spread"],
        dist_threshold=context["dist_threshold"],
        radial_threshold=context["radial_threshold"],
    )

    span_penalty = metrics["angular_span"] / 180.0
    pair_penalty = metrics["max_pairwise"] / max(context["dist_threshold"], 1e-9)
    radial_penalty = metrics["radial_spread"] / max(context["radial_threshold"], 1e-9)
    cost_penalty = cost / max(singleton_total, 1e-9)
    normalized_savings = savings / max(context["median_nearest_neighbor_distance"], 1e-9)

    score = (
        weights.get("density", 1.0) * density_score
        + weights.get("savings", 1.0) * normalized_savings
        - weights.get("span_penalty", 0.5) * span_penalty
        - weights.get("pair_penalty", 0.5) * pair_penalty
        - weights.get("radial_penalty", 0.25) * radial_penalty
        - weights.get("cost_penalty", 0.15) * cost_penalty
    )

    return {
        "subset": canonical_subset(subset),
        "order": list(order),
        "cost": float(cost),
        "size": len(subset),
        "score": float(score),
        "density_score": float(density_score),
        "savings": float(savings),
        "angular_span": float(metrics["angular_span"]),
        "max_pairwise": float(metrics["max_pairwise"]),
        "radial_spread": float(metrics["radial_spread"]),
    }


def subset_matches_promising_data(subset, promising_data=None):
    if not promising_data:
        return True

    subset = canonical_subset(subset)
    if len(subset) <= 2:
        return True

    subset_set = set(subset)
    seed_pairs = set(promising_data.get("seed_pairs", set()))
    seed_groups = [set(group) for group in promising_data.get("seed_groups", [])]
    top_customers = set(promising_data.get("top_customers", []))

    pair_hits = sum(
        1
        for pair in combinations(subset, 2)
        if canonical_subset(pair) in seed_pairs
    )

    if len(subset) == 3 and pair_hits >= 1:
        return True
    if len(subset) >= 4 and pair_hits >= 2:
        return True

    for group in seed_groups:
        if len(subset_set & group) >= min(len(subset) - 1, len(group)):
            return True

    if top_customers and len(subset_set & top_customers) >= min(len(subset) - 1, len(top_customers)):
        return True

    return False


def generate_candidate_subsets(
    context,
    C,
    config=None,
    allowed_sizes=None,
    promising_data=None,
    excluded_subsets=None,
):
    config = dict(config or {})
    customers = context["customers"]
    if allowed_sizes is None:
        allowed_sizes = list(range(1, min(C, len(customers)) + 1))

    allowed_sizes = sorted(
        set(size for size in allowed_sizes if 1 <= size <= min(C, len(customers)))
    )
    excluded_subsets = {
        canonical_subset(subset) for subset in (excluded_subsets or set())
    }

    use_neighbor_generator = config.get("use_neighbor_generator", True)
    generated = []
    seen = set()

    for size in allowed_sizes:
        if size == 1:
            for customer in customers:
                subset = (customer,)
                if subset not in excluded_subsets and subset not in seen:
                    seen.add(subset)
                    generated.append(subset)
            continue

        if use_neighbor_generator:
            for anchor in customers:
                neighbor_pool = [node for node in context["nearest"].get(anchor, []) if node != anchor]
                if len(neighbor_pool) < size - 1:
                    continue
                for combo in combinations(neighbor_pool, size - 1):
                    subset = canonical_subset((anchor,) + combo)
                    if subset in excluded_subsets or subset in seen:
                        continue
                    if not subset_matches_promising_data(subset, promising_data):
                        continue
                    seen.add(subset)
                    generated.append(subset)
        else:
            for subset in combinations(customers, size):
                subset = canonical_subset(subset)
                if subset in excluded_subsets or subset in seen:
                    continue
                if not subset_matches_promising_data(subset, promising_data):
                    continue
                seen.add(subset)
                generated.append(subset)

    return dedupe_canonical_subsets(generated)


def sort_route_pool(route_pool):
    return sorted(
        route_pool,
        key=lambda route: (
            -route["score"],
            route["cost"],
            route["size"],
            canonical_subset(route["subset"]),
        ),
    )


def merge_route_entries(existing_pool, new_routes):
    merged = {}
    for route in sort_route_pool(list(existing_pool) + list(new_routes)):
        key = canonical_subset(route["subset"])
        if key not in merged:
            route_copy = dict(route)
            route_copy["subset"] = key
            route_copy["order"] = list(route["order"])
            merged[key] = route_copy
    return list(merged.values())


def is_near_duplicate_route(route, selected_routes, config=None):
    config = dict(config or {})
    if route["size"] <= 1 or not config.get("use_diversity_guard", True):
        return False

    threshold = float(config.get("diversity_jaccard_threshold", 0.80))
    route_set = set(route["subset"])

    for other in selected_routes:
        if other["size"] <= 1:
            continue
        other_set = set(other["subset"])
        overlap = len(route_set & other_set)
        union = len(route_set | other_set)
        if union > 0 and overlap / union >= threshold:
            return True
        if route["size"] == other["size"] and route["size"] >= 3 and overlap >= route["size"] - 1:
            return True
    return False


def apply_guarded_route_pool_cap(route_pool, customers, config=None):
    config = dict(config or {})
    ranked = sort_route_pool(route_pool)
    if not ranked:
        return [], {"after_guarded_cap": 0, "preserved_routes": 0}

    use_pool_cap = config.get("use_pool_cap", True)
    max_pool_size = config.get("max_route_pool_size", None)
    pair_keep = int(config.get("m_pairs_per_customer", 2))
    higher_keep = int(config.get("m_higher_per_customer", 1))

    selected = []
    selected_keys = set()

    def add_route(route, apply_diversity_guard=False):
        key = canonical_subset(route["subset"])
        if key in selected_keys:
            return False
        if apply_diversity_guard and is_near_duplicate_route(route, selected, config):
            return False
        selected.append(dict(route))
        selected_keys.add(key)
        return True

    for route in ranked:
        if route["size"] == 1:
            add_route(route)

    pair_candidates = {
        customer: [
            route for route in ranked
            if route["size"] == 2 and customer in route["subset"]
        ]
        for customer in customers
    }
    for customer in customers:
        for route in pair_candidates[customer][:pair_keep]:
            add_route(route)

    higher_candidates = {
        customer: [
            route for route in ranked
            if route["size"] >= 3 and customer in route["subset"]
        ]
        for customer in customers
    }
    for customer in customers:
        for route in higher_candidates[customer][:higher_keep]:
            add_route(route)

    preserved_routes = len(selected)

    if not use_pool_cap:
        for route in ranked:
            add_route(route)
    else:
        limit = None if max_pool_size in (None, 0) else int(max_pool_size)
        if limit is None:
            for route in ranked:
                add_route(route)
        else:
            for route in ranked:
                if len(selected) >= limit:
                    break
                add_route(route, apply_diversity_guard=True)

    final_pool = sort_route_pool(selected)
    for var_idx, route in enumerate(final_pool):
        route["var_idx"] = var_idx

    return final_pool, {
        "after_guarded_cap": len(final_pool),
        "preserved_routes": preserved_routes,
    }


def generate_route_pool(
    nodes,
    Nv,
    C,
    config=None,
    allowed_sizes=None,
    promising_data=None,
    existing_pool=None,
):
    config = dict(config or {})
    context = build_route_generation_context(nodes, config)
    D = context["D"]
    customers = context["customers"]

    if allowed_sizes is None:
        allowed_sizes = list(range(1, min(C, len(customers)) + 1))

    existing_pool = list(existing_pool or [])
    existing_subsets = {canonical_subset(route["subset"]) for route in existing_pool}

    candidate_subsets = generate_candidate_subsets(
        context=context,
        C=C,
        config=config,
        allowed_sizes=allowed_sizes,
        promising_data=promising_data,
        excluded_subsets=existing_subsets,
    )

    new_routes = []
    for subset in candidate_subsets:
        keep_subset, metrics = subset_passes_geometric_pruning(subset, context, config)
        if not keep_subset:
            continue
        order, cost = exact_subset_route_cost(subset, D)
        new_routes.append(score_route_entry(subset, order, cost, metrics, context, config))

    combined_pool = merge_route_entries(existing_pool, new_routes)
    final_pool, cap_stats = apply_guarded_route_pool_cap(combined_pool, customers, config)

    final_keys = {canonical_subset(route["subset"]) for route in final_pool}
    stats = {
        "allowed_sizes": sorted(
            set(size for size in allowed_sizes if 1 <= size <= min(C, len(customers)))
        ),
        "total_candidate_subsets_before_pruning": count_candidate_subsets(
            len(customers),
            allowed_sizes,
        ),
        "after_neighbor_restriction": len(candidate_subsets),
        "after_geometric_pruning": len(new_routes),
        "after_guarded_cap": cap_stats["after_guarded_cap"],
        "generated_new_routes": len(new_routes),
        "new_routes_added": len(final_keys - existing_subsets),
        "median_nearest_neighbor_distance": context["median_nearest_neighbor_distance"],
        "dist_threshold": context["dist_threshold"],
        "radial_threshold": context["radial_threshold"],
        "final_count_by_size": count_routes_by_size(final_pool),
        "preserved_routes": cap_stats["preserved_routes"],
    }
    return final_pool, D, stats, context


def print_route_pool_summary(route_pool, stats=None, label="Route pool"):
    print(label)
    if stats is not None:
        print(f"  total candidate subsets before pruning: {stats['total_candidate_subsets_before_pruning']}")
        print(f"  after neighbor restriction: {stats['after_neighbor_restriction']}")
        print(f"  after geometric pruning: {stats['after_geometric_pruning']}")
        print(f"  after guarded cap: {stats['after_guarded_cap']}")
        print(f"  new routes added: {stats.get('new_routes_added', 0)}")
        print(f"  median nearest-neighbor distance: {stats['median_nearest_neighbor_distance']:.4f}")
        print(f"  adaptive pairwise threshold: {stats['dist_threshold']:.4f}")
        print(f"  adaptive radial threshold: {stats['radial_threshold']:.4f}")
    print("  final count by subset size:")
    for size, count in count_routes_by_size(route_pool).items():
        print(f"    size {size}: {count}")
    print(f"  total route variables: {len(route_pool)}")


def extract_promising_neighborhoods(route_pool, solver_result, max_seed_groups=12, max_seed_pairs=24):
    pair_scores = Counter()
    group_scores = Counter()
    customer_scores = Counter()

    def add_subset_support(subset, weight):
        subset = canonical_subset(subset)
        if len(subset) <= 1 or weight <= 0.0:
            return
        group_scores[subset] += weight
        for customer in subset:
            customer_scores[customer] += weight
        for pair in combinations(subset, 2):
            pair_scores[canonical_subset(pair)] += weight

    for rank, sample in enumerate(solver_result.get("top_feasible_samples", [])[:8]):
        weight = 4.0 / (1.0 + rank)
        for subset in sample.get("selected_subsets", []):
            add_subset_support(subset, weight)

    for rank, sample in enumerate(solver_result.get("elite_samples", [])[:8]):
        weight = 2.0 / (1.0 + rank)
        for subset in sample.get("selected_subsets", []):
            add_subset_support(subset, weight)

    marginals = np.asarray(solver_result.get("route_marginals", []), dtype=float)
    if marginals.size:
        ranked_indices = list(np.argsort(-marginals))[: min(len(route_pool), 16)]
        for idx in ranked_indices:
            weight = float(marginals[int(idx)])
            if weight <= 0.0:
                continue
            add_subset_support(route_pool[int(idx)]["subset"], weight)

    seed_pairs = {pair for pair, _ in pair_scores.most_common(max_seed_pairs)}
    seed_groups = [subset for subset, _ in group_scores.most_common(max_seed_groups)]
    top_customers = [
        customer
        for customer, _ in customer_scores.most_common(max(4, min(10, len(customer_scores))))
    ]

    return {
        "seed_pairs": seed_pairs,
        "seed_groups": seed_groups,
        "top_customers": top_customers,
        "pair_scores": pair_scores,
        "group_scores": group_scores,
        "customer_scores": customer_scores,
    }


def print_promising_summary(promising_data, label="Promising neighborhoods"):
    if not promising_data:
        print(f"{label}: none")
        return
    print(
        f"{label}: {len(promising_data.get('seed_pairs', []))} seed pairs, "
        f"{len(promising_data.get('seed_groups', []))} seed groups"
    )
    if promising_data.get("seed_groups"):
        preview = ", ".join(str(group) for group in promising_data["seed_groups"][:5])
        print(f"  top groups: {preview}")


In [ ]:
# ── Phase 2: Route-Pool Selection QUBO ──

def build_subset_qubo(route_pool, n_customers, Nv, A=None, B=None):
    """
    Build the set-partitioning QUBO:

      E(y) = sum_S c_S y_S
           + A * sum_i (1 - sum_{S contains i} y_S)^2
           + B * (Nv - sum_S y_S)^2

    where each binary variable y_S selects one exact depot-to-depot route.
    """
    n_vars = len(route_pool)

    if A is None:
        A = 2.0 * max(r["cost"] for r in route_pool)
    if B is None:
        B = 2.0 * A

    Q = defaultdict(float)
    constant = A * n_customers + B * (Nv ** 2)

    # Objective: route cost
    for i, route in enumerate(route_pool):
        Q[(i, i)] += route["cost"]

    # Coverage penalty: each customer covered exactly once
    covers = defaultdict(list)
    for i, route in enumerate(route_pool):
        for cust in route["subset"]:
            covers[cust].append(i)

    for cust, idxs in covers.items():
        # (1 - sum y)^2 = 1 - sum y + 2 sum_{a<b} y_a y_b   since y^2 = y
        for i in idxs:
            Q[(i, i)] += -A
        for a in range(len(idxs)):
            for b in range(a + 1, len(idxs)):
                i, j = idxs[a], idxs[b]
                if i > j:
                    i, j = j, i
                Q[(i, j)] += 2.0 * A

    # Route-count penalty: use exactly Nv selected routes
    # (Nv - sum y)^2 = Nv^2 + (1 - 2Nv) sum y + 2 sum_{a<b} y_a y_b
    for i in range(n_vars):
        Q[(i, i)] += B * (1.0 - 2.0 * Nv)

    for i in range(n_vars):
        for j in range(i + 1, n_vars):
            Q[(i, j)] += 2.0 * B

    return dict(Q), float(constant), float(A), float(B)


def qubo_dict_to_ising(Q, n):
    """
    Convert QUBO dict to Ising coefficients using x = (1 - Z)/2.
    Returns:
      h : local Z coefficients
      J : ZZ couplings
      constant : Ising constant offset
    """
    constant = 0.0
    h = np.zeros(n)
    J = {}

    for (i, j), q in Q.items():
        if i == j:
            constant += q / 2.0
            h[i]     -= q / 2.0
        else:
            constant += q / 4.0
            h[i]     -= q / 4.0
            h[j]     -= q / 4.0
            J[(i, j)] = J.get((i, j), 0.0) + q / 4.0

    return h, J, float(constant)


def decode_route_selection(bitstring_or_bits, route_pool, Nv, n_customers):
    """
    Decode selected route-subset variables into explicit depot routes.
    Returns (routes, total_cost) if feasible, else (None, inf).
    """
    if isinstance(bitstring_or_bits, str):
        x = bitstring_to_array(bitstring_or_bits, len(route_pool))
    else:
        x = np.asarray(bitstring_or_bits, dtype=int)

    chosen = [route_pool[i] for i, bit in enumerate(x) if bit == 1]

    if len(chosen) != Nv:
        return None, float('inf')

    visited = []
    for route in chosen:
        visited.extend(route["subset"])

    if len(visited) != n_customers:
        return None, float('inf')
    if len(set(visited)) != n_customers:
        return None, float('inf')

    routes = [[0] + route["order"] + [0] for route in chosen]
    total_cost = float(sum(route["cost"] for route in chosen))
    return routes, total_cost

In [ ]:
# -- Phase 3: BF-DCQO (Notebook Scaffold) --
#
# Important:
# This notebook uses a BF-DCQO-style iterative bias-update loop, but the
# function `build_bf_dcqo_surrogate_circuit(...)` is the ONE place you should
# swap in a more paper-faithful first-order counterdiabatic block later.
#
# Everything else in the notebook -- route encoding, QUBO build, elite-sample
# bias updates, decoding, feasibility, and BBB refinement -- can stay the same.


def schedule_lambda(step, total_steps):
    u = step / total_steps
    return float(np.sin(np.pi * (np.sin(np.pi * u / 2.0) ** 2) / 2.0) ** 2)


def build_bf_dcqo_surrogate_circuit(h, J, hb, p=3, theta_scale=1.0, theta_cutoff=0.0):
    n = len(h)
    qc = QuantumCircuit(n, n)
    qc.h(range(n))

    dt = 1.0 / max(1, p)

    for s in range(1, p + 1):
        lam = schedule_lambda(s, p)

        mixer_angle = 2.0 * dt * (1.0 - lam)
        for q in range(n):
            qc.rx(mixer_angle, q)

        for q in range(n):
            theta = 2.0 * dt * (lam * h[q] + hb[q])
            if abs(theta) > theta_cutoff:
                qc.rz(theta_scale * theta, q)

        for (i, j), coef in J.items():
            theta = 2.0 * dt * lam * coef
            if abs(theta) > theta_cutoff:
                qc.rzz(theta_scale * theta, i, j)

    qc.measure(range(n), range(n))
    return qc


def sample_counts(circuit, shots=1024, optimization_level=1):
    backend = AerSimulator()
    tcirc = transpile(circuit, backend=backend, optimization_level=optimization_level)
    result = backend.run(tcirc, shots=shots).result()
    counts = result.get_counts()

    gate_count = sum(tcirc.count_ops().values())
    n_qubits = tcirc.num_qubits
    return counts, gate_count, n_qubits


def elite_shots(counts, Q, constant, alpha=0.02):
    expanded = []
    for bitstring, cnt in counts.items():
        e = energy_of_bitstring(bitstring, Q, constant)
        expanded.extend([(bitstring, e)] * cnt)

    expanded.sort(key=lambda item: item[1])
    keep = max(1, int(math.ceil(alpha * len(expanded))))
    return expanded[:keep]


def compute_bias_update(counts, Q, constant, n, alpha=0.02, mode="unsigned", kappa=5.0):
    elite = elite_shots(counts, Q, constant, alpha=alpha)

    z_sum = np.zeros(n)
    best_bits = None
    best_energy = float("inf")

    for bitstring, energy in elite:
        x = bitstring_to_array(bitstring, n)
        z = np.where(x == 0, 1.0, -1.0)
        z_sum += z

        if energy < best_energy:
            best_energy = energy
            best_bits = x.copy()

    mags = z_sum / max(1, len(elite))

    if best_bits is None:
        return np.zeros(n), None, float("inf")

    if mode == "unsigned":
        direction = np.where(best_bits == 0, 1.0, -1.0)
        hb = direction * np.abs(mags)
    else:
        hb = kappa * mags

    return hb, best_bits, float(best_energy)


def run_bf_dcqo_generic_qubo(
    Q,
    constant,
    p=3,
    bias_iters=6,
    shots=1024,
    alpha=0.02,
    theta_scale=1.0,
    theta_cutoff=0.0,
    kappa_final=5.0,
):
    n = infer_num_vars_from_qubo(Q)
    h, J, ising_constant = qubo_dict_to_ising(Q, n)

    hb = np.zeros(n)
    history = []
    iteration_counts = []

    best_energy = float("inf")
    best_bits = None
    total_gates = 0
    total_time = 0.0
    max_qubits = n

    for it in range(bias_iters):
        mode = "signed" if it == bias_iters - 1 else "unsigned"

        t0 = time.time()
        qc = build_bf_dcqo_surrogate_circuit(
            h,
            J,
            hb,
            p=p,
            theta_scale=theta_scale,
            theta_cutoff=theta_cutoff,
        )
        counts, gate_count, n_qubits = sample_counts(qc, shots=shots)
        elapsed = time.time() - t0

        iteration_counts.append(counts)
        total_gates += gate_count
        total_time += elapsed
        max_qubits = max(max_qubits, n_qubits)

        iter_best_energy = float("inf")
        iter_best_bits = None
        for bitstring in counts:
            e = energy_of_bitstring(bitstring, Q, constant)
            if e < iter_best_energy:
                iter_best_energy = e
                iter_best_bits = bitstring_to_array(bitstring, n)

        if iter_best_energy < best_energy:
            best_energy = iter_best_energy
            best_bits = iter_best_bits.copy()

        hb, elite_best_bits, elite_best_energy = compute_bias_update(
            counts,
            Q,
            constant,
            n,
            alpha=alpha,
            mode=mode,
            kappa=kappa_final,
        )

        history.append(
            {
                "iter": it + 1,
                "mode": mode,
                "raw_best_energy": float(iter_best_energy),
                "elite_best_energy": float(elite_best_energy),
                "gate_count": int(gate_count),
                "shots": int(shots),
            }
        )

    return {
        "best_bits": best_bits,
        "best_energy": float(best_energy),
        "history": history,
        "iteration_counts": iteration_counts,
        "last_counts": iteration_counts[-1] if iteration_counts else {},
        "total_gates": int(total_gates),
        "total_time": float(total_time),
        "max_qubits": int(max_qubits),
    }


def run_bf_dcqo_subset_solver(
    route_pool,
    Q,
    constant,
    Nv,
    n_customers,
    p=3,
    bias_iters=6,
    shots=1024,
    alpha=0.02,
    theta_scale=1.0,
    theta_cutoff=0.0,
    kappa_final=5.0,
):
    base = run_bf_dcqo_generic_qubo(
        Q=Q,
        constant=constant,
        p=p,
        bias_iters=bias_iters,
        shots=shots,
        alpha=alpha,
        theta_scale=theta_scale,
        theta_cutoff=theta_cutoff,
        kappa_final=kappa_final,
    )

    n = len(route_pool)
    aggregate_counts = Counter()
    feasible_records = {}

    best_routes = None
    best_cost = float("inf")
    best_feasible_bits = None
    total_feasible_samples = 0
    total_samples = 0

    for iter_idx, counts in enumerate(base["iteration_counts"], start=1):
        for bitstring, cnt in counts.items():
            aggregate_counts[bitstring] += cnt
            total_samples += cnt

            routes, cost = decode_route_selection(bitstring, route_pool, Nv, n_customers)
            if routes is None:
                continue

            total_feasible_samples += cnt
            bits = bitstring_to_array(bitstring, n)

            record = feasible_records.setdefault(
                bitstring,
                {
                    "bitstring": bitstring,
                    "bits": bits,
                    "count": 0,
                    "cost": float(cost),
                    "routes": routes,
                    "selected_subsets": [],
                    "iteration": iter_idx,
                },
            )
            record["count"] += cnt
            record["selected_subsets"] = [
                canonical_subset(route_pool[idx]["subset"])
                for idx, bit in enumerate(bits)
                if bit == 1
            ]

            if cost < best_cost:
                best_cost = cost
                best_routes = routes
                best_feasible_bits = bits.copy()

    feasible_rate = total_feasible_samples / max(1, total_samples)
    route_marginals = counts_to_marginals(aggregate_counts, n) if n else np.zeros(0)

    top_feasible_samples = []
    for item in sorted(
        feasible_records.values(),
        key=lambda row: (row["cost"], -row["count"], row["bitstring"]),
    )[:12]:
        top_feasible_samples.append(
            {
                "bitstring": item["bitstring"],
                "count": int(item["count"]),
                "cost": float(item["cost"]),
                "routes": item["routes"],
                "selected_subsets": item["selected_subsets"],
                "iteration": int(item["iteration"]),
            }
        )

    elite_samples = []
    for bitstring, cnt in sorted(
        aggregate_counts.items(),
        key=lambda item: (
            energy_of_bitstring(item[0], Q, constant),
            -item[1],
            item[0],
        ),
    )[:12]:
        bits = bitstring_to_array(bitstring, n)
        chosen_indices = [idx for idx, bit in enumerate(bits) if bit == 1]
        elite_samples.append(
            {
                "bitstring": bitstring,
                "count": int(cnt),
                "energy": float(energy_of_bitstring(bitstring, Q, constant)),
                "selected_route_indices": chosen_indices,
                "selected_subsets": [
                    canonical_subset(route_pool[idx]["subset"])
                    for idx in chosen_indices
                ],
            }
        )

    return {
        **base,
        "routes": best_routes,
        "cost": float(best_cost),
        "best_feasible_bits": best_feasible_bits,
        "feasible_rate": float(feasible_rate),
        "aggregate_counts": dict(aggregate_counts),
        "route_marginals": route_marginals,
        "top_feasible_samples": top_feasible_samples,
        "elite_samples": elite_samples,
    }


In [ ]:
# -- Phase 4: BBB-DCQO Selective Refinement --
#
# Idea:
#   1. Run BF-DCQO first
#   2. Compute route-variable marginals from the final sample distribution
#   3. Fix obvious 0/1 variables
#   4. Branch only on the most uncertain variables
#   5. Solve each reduced branch with a shallower BF-DCQO run
#   6. Keep the best feasible full solution


def compute_marginals(counts, n):
    return counts_to_marginals(counts, n)


def reduce_qubo_with_fixed(Q, constant, n, fixed):
    free_vars = [i for i in range(n) if i not in fixed]
    new_pos = {old: new for new, old in enumerate(free_vars)}

    Q_red = defaultdict(float)
    constant_red = float(constant)

    for (i, j), q in Q.items():
        vi = fixed.get(i, None)
        vj = fixed.get(j, None)

        if i == j:
            if vi is not None:
                constant_red += q * vi
            else:
                Q_red[(new_pos[i], new_pos[i])] += q
            continue

        if vi is not None and vj is not None:
            constant_red += q * vi * vj
        elif vi is not None and vj is None:
            Q_red[(new_pos[j], new_pos[j])] += q * vi
        elif vi is None and vj is not None:
            Q_red[(new_pos[i], new_pos[i])] += q * vj
        else:
            a, b = new_pos[i], new_pos[j]
            if a > b:
                a, b = b, a
            Q_red[(a, b)] += q

    return dict(Q_red), float(constant_red), free_vars


def expand_reduced_bits(reduced_bits, free_vars, fixed, n):
    full = np.zeros(n, dtype=int)
    for idx, val in fixed.items():
        full[idx] = val
    for pos, old_idx in enumerate(free_vars):
        full[old_idx] = reduced_bits[pos]
    return full


def conflicts_if_set_one(var_idx, fixed, route_pool):
    candidate = set(route_pool[var_idx]["subset"])
    for idx, val in fixed.items():
        if val == 1:
            chosen = set(route_pool[idx]["subset"])
            if candidate & chosen:
                return True
    return False


def run_bbb_dcqo_refinement(
    bf_result,
    route_pool,
    Q,
    constant,
    Nv,
    n_customers,
    p_small=2,
    bias_iters_small=3,
    shots_small=512,
    alpha=0.02,
    high_fix=0.95,
    low_fix=0.05,
    max_branch_vars=6,
    max_nodes=16,
):
    n = len(route_pool)
    counts = bf_result["last_counts"]

    if not counts:
        return {
            "routes": bf_result["routes"],
            "cost": bf_result["cost"],
            "source": "BF-DCQO (no BBB counts available)",
            "branches_tried": 0,
        }

    marginals = compute_marginals(counts, n)

    fixed_base = {i: 0 for i, p in enumerate(marginals) if p <= low_fix}

    high_candidates = [i for i, p in enumerate(marginals) if p >= high_fix]
    high_candidates.sort(key=lambda i: -marginals[i])

    for idx in high_candidates:
        if idx in fixed_base:
            continue
        if not conflicts_if_set_one(idx, fixed_base, route_pool):
            fixed_base[idx] = 1

    uncertain = [i for i in range(n) if i not in fixed_base]
    uncertain.sort(key=lambda i: abs(marginals[i] - 0.5))
    uncertain = uncertain[:max_branch_vars]

    best_routes = bf_result["routes"]
    best_cost = bf_result["cost"]
    branches_tried = 0

    stack = [fixed_base]

    while stack and branches_tried < max_nodes:
        fixed = stack.pop()

        next_var = None
        for idx in uncertain:
            if idx not in fixed:
                next_var = idx
                break

        if next_var is None:
            Q_red, constant_red, free_vars = reduce_qubo_with_fixed(Q, constant, n, fixed)

            if len(free_vars) == 0:
                full_bits = np.array([fixed.get(i, 0) for i in range(n)], dtype=int)
                routes, cost = decode_route_selection(full_bits, route_pool, Nv, n_customers)
                if routes is not None and cost < best_cost:
                    best_routes, best_cost = routes, cost
                branches_tried += 1
                continue

            reduced = run_bf_dcqo_generic_qubo(
                Q=Q_red,
                constant=constant_red,
                p=p_small,
                bias_iters=bias_iters_small,
                shots=shots_small,
                alpha=alpha,
                theta_scale=1.0,
                theta_cutoff=0.0,
                kappa_final=5.0,
            )

            for counts_red in reduced["iteration_counts"]:
                for bitstring in counts_red:
                    reduced_bits = bitstring_to_array(bitstring, len(free_vars))
                    full_bits = expand_reduced_bits(reduced_bits, free_vars, fixed, n)
                    routes, cost = decode_route_selection(full_bits, route_pool, Nv, n_customers)

                    if routes is not None and cost < best_cost:
                        best_routes, best_cost = routes, cost

            branches_tried += 1
            continue

        prob = marginals[next_var]
        preferred = 1 if prob >= 0.5 else 0
        other = 1 - preferred

        fixed_pref = dict(fixed)
        if not (preferred == 1 and conflicts_if_set_one(next_var, fixed_pref, route_pool)):
            fixed_pref[next_var] = preferred
            stack.append(fixed_pref)

        fixed_alt = dict(fixed)
        if not (other == 1 and conflicts_if_set_one(next_var, fixed_alt, route_pool)):
            fixed_alt[next_var] = other
            stack.append(fixed_alt)

    source = "BBB-DCQO refinement" if best_cost < bf_result["cost"] else "BF-DCQO"
    return {
        "routes": best_routes,
        "cost": float(best_cost),
        "source": source,
        "branches_tried": int(branches_tried),
    }


In [ ]:
# -- Instance Library --

instances = {
    1: {"Nv": 2, "C": 5, "nodes": [(0, 0), (-2, 2), (-5, 8), (2, 3)]},
    2: {"Nv": 2, "C": 2, "nodes": [(0, 0), (-2, 2), (-5, 8), (2, 3)]},
    3: {"Nv": 3, "C": 2, "nodes": [(0, 0), (-2, 2), (-5, 8), (2, 3), (5, 7), (2, 4), (2, -3)]},
    4: {
        "Nv": 4,
        "C": 3,
        "nodes": [
            (0, 0), (-2, 2), (-5, 8), (6, 3), (4, 4), (3, 2),
            (0, 2), (-2, 3), (-4, 3), (2, 3), (2, 7), (-2, 5), (-1, 4),
        ],
    },
}


# -- Configuration --
read_from_file = False
selected_file_id = "instance_1"
selected_id = 4

use_bbb_refinement = True

# BF-DCQO params
p = 3
bias_iters = 6
shots = 1024
alpha = 0.02

# BBB refinement params
p_small = 2
bias_iters_small = 3
shots_small = 512

route_pool_mode = "staged"
use_geometric_pruning = True
use_neighbor_generator = True
use_pool_cap = True

k_nearest = 6
beta_dist = 3.5
radial_spread_threshold = 2.25

angle_threshold_by_size = {
    2: 180.0,
    3: 130.0,
    4: 95.0,
    5: 80.0,
}

max_route_pool_size = None
m_pairs_per_customer = 2
m_higher_per_customer = 1
use_diversity_guard = True
diversity_jaccard_threshold = 0.80

score_weights = {
    "density": 1.0,
    "savings": 1.0,
    "span_penalty": 0.5,
    "pair_penalty": 0.5,
}


if read_from_file:
    with open("instances_2.json", "r") as f:
        all_instances = json.load(f)

    instance_cfg = next(
        (inst for inst in all_instances if inst["instance_id"] == selected_file_id),
        None,
    )
    assert instance_cfg is not None, f"Instance '{selected_file_id}' not found"

    Nv = instance_cfg["Nv"]
    C = instance_cfg["C"]

    raw_nodes = [(c["x"], c["y"]) for c in instance_cfg["customers"]]
    nodes = raw_nodes if raw_nodes[0] == (0, 0) else [(0, 0)] + raw_nodes
    selected_id = selected_file_id
else:
    instance_cfg = instances[selected_id]
    Nv = instance_cfg["Nv"]
    C = instance_cfg["C"]
    nodes = instance_cfg["nodes"]

print(f"Loaded Instance {selected_id}: {len(nodes) - 1} customers, {Nv} vehicles, capacity {C}")

route_pool_config = {
    "route_pool_mode": route_pool_mode,
    "use_geometric_pruning": use_geometric_pruning,
    "use_neighbor_generator": use_neighbor_generator,
    "use_pool_cap": use_pool_cap,
    "k_nearest": k_nearest,
    "beta_dist": beta_dist,
    "radial_spread_threshold": radial_spread_threshold,
    "angle_threshold_by_size": angle_threshold_by_size,
    "max_route_pool_size": max_route_pool_size,
    "m_pairs_per_customer": m_pairs_per_customer,
    "m_higher_per_customer": m_higher_per_customer,
    "use_diversity_guard": use_diversity_guard,
    "diversity_jaccard_threshold": diversity_jaccard_threshold,
    "score_weights": score_weights,
}

print(f"Route-pool mode: {route_pool_config['route_pool_mode']}")
print(
    "Route-pool settings: "
    f"k_nearest={route_pool_config['k_nearest']}, "
    f"beta_dist={route_pool_config['beta_dist']}, "
    f"radial_spread_threshold={route_pool_config['radial_spread_threshold']}, "
    f"max_route_pool_size={route_pool_config['max_route_pool_size']}"
)


In [ ]:
# -- Main Run --

def print_bf_history(history):
    for row in history:
        print(
            f"  iter {row['iter']:>2} | "
            f"mode={row['mode']:<8} | "
            f"raw_best_E={row['raw_best_energy']:>10.3f} | "
            f"elite_best_E={row['elite_best_energy']:>10.3f} | "
            f"gates={row['gate_count']}"
        )


def solve_route_pool_once(
    route_pool,
    D,
    Nv,
    n_customers,
    use_bbb=False,
    p=3,
    bias_iters=6,
    shots=1024,
    alpha=0.02,
    p_small=2,
    bias_iters_small=3,
    shots_small=512,
):
    Q, constant, A, B = build_subset_qubo(
        route_pool=route_pool,
        n_customers=n_customers,
        Nv=Nv,
    )

    bf_result = run_bf_dcqo_subset_solver(
        route_pool=route_pool,
        Q=Q,
        constant=constant,
        Nv=Nv,
        n_customers=n_customers,
        p=p,
        bias_iters=bias_iters,
        shots=shots,
        alpha=alpha,
        theta_scale=1.0,
        theta_cutoff=0.0,
        kappa_final=5.0,
    )

    selected_routes = bf_result["routes"]
    selected_cost = bf_result["cost"]
    selected_source = "BF-DCQO"
    bbb_result = None

    if use_bbb:
        bbb_result = run_bbb_dcqo_refinement(
            bf_result=bf_result,
            route_pool=route_pool,
            Q=Q,
            constant=constant,
            Nv=Nv,
            n_customers=n_customers,
            p_small=p_small,
            bias_iters_small=bias_iters_small,
            shots_small=shots_small,
            alpha=alpha,
            high_fix=0.95,
            low_fix=0.05,
            max_branch_vars=6,
            max_nodes=16,
        )

        if bbb_result["routes"] is not None and bbb_result["cost"] < selected_cost:
            selected_routes = bbb_result["routes"]
            selected_cost = bbb_result["cost"]
            selected_source = bbb_result["source"]

    return {
        "route_pool": route_pool,
        "D": D,
        "Q": Q,
        "constant": constant,
        "A": A,
        "B": B,
        "bf_result": bf_result,
        "bbb_result": bbb_result,
        "routes": selected_routes,
        "cost": float(selected_cost),
        "source": selected_source,
    }


def print_stage_summary(stage_label, pool_stats, stage_result):
    bf_result = stage_result["bf_result"]
    best_cost = stage_result["cost"]
    best_cost_text = "inf" if not np.isfinite(best_cost) else f"{best_cost:.4f}"

    print(f"\n{'=' * 56}")
    print(stage_label)
    print(f"{'=' * 56}")
    print_route_pool_summary(stage_result["route_pool"], pool_stats, label="Route-pool summary")
    print("Penalty coefficients:")
    print(f"  A = {stage_result['A']:.4f}")
    print(f"  B = {stage_result['B']:.4f}")
    print("BF-DCQO history:")
    print_bf_history(bf_result["history"])
    print(f"  route variables: {len(stage_result['route_pool'])}")
    print(f"  best feasible cost: {best_cost_text}")
    print(f"  feasible sample rate: {bf_result['feasible_rate']:.4f}")
    print(f"  new routes added: {pool_stats.get('new_routes_added', 0)}")
    print(f"  max qubits: {bf_result['max_qubits']}")
    print(f"  total gates: {bf_result['total_gates']}")
    print(f"  total time: {bf_result['total_time']:.2f}s")
    if stage_result["bbb_result"] is not None:
        print(f"  BBB branches tried: {stage_result['bbb_result']['branches_tried']}")
        print(f"  selected source: {stage_result['source']}")


def staged_size_plan(C):
    plan = []
    if C >= 1:
        plan.append(("Stage 1", [size for size in [1, 2] if size <= C]))
    if C >= 3:
        plan.append(("Stage 2", [3]))
    if C >= 4:
        plan.append(("Stage 3", list(range(4, C + 1))))
    return [(label, sizes) for label, sizes in plan if sizes]


def run_single_instance_solver(
    nodes,
    Nv,
    C,
    route_pool_config,
    use_bbb_refinement=True,
    verbose=True,
):
    n_customers = len(nodes) - 1
    mode = route_pool_config.get("route_pool_mode", "oneshot")
    stage_records = []
    final_result = None
    route_pool = []
    D = build_distance_matrix(nodes)

    if mode == "oneshot":
        route_pool, D, pool_stats, context = generate_route_pool(
            nodes=nodes,
            Nv=Nv,
            C=C,
            config=route_pool_config,
            allowed_sizes=list(range(1, min(C, n_customers) + 1)),
        )
        final_result = solve_route_pool_once(
            route_pool=route_pool,
            D=D,
            Nv=Nv,
            n_customers=n_customers,
            use_bbb=use_bbb_refinement,
            p=p,
            bias_iters=bias_iters,
            shots=shots,
            alpha=alpha,
            p_small=p_small,
            bias_iters_small=bias_iters_small,
            shots_small=shots_small,
        )
        stage_records.append(
            {
                "label": "Oneshot",
                "sizes": list(range(1, min(C, n_customers) + 1)),
                "pool_stats": pool_stats,
                "result": final_result,
            }
        )
        if verbose:
            print_stage_summary("Oneshot solve", pool_stats, final_result)
    else:
        promising_data = None
        current_pool = []

        for stage_index, (stage_label, sizes) in enumerate(staged_size_plan(C), start=1):
            route_pool, D, pool_stats, context = generate_route_pool(
                nodes=nodes,
                Nv=Nv,
                C=C,
                config=route_pool_config,
                allowed_sizes=sizes,
                promising_data=promising_data,
                existing_pool=current_pool,
            )
            current_pool = route_pool

            final_stage = stage_index == len(staged_size_plan(C))
            stage_result = solve_route_pool_once(
                route_pool=current_pool,
                D=D,
                Nv=Nv,
                n_customers=n_customers,
                use_bbb=use_bbb_refinement and final_stage,
                p=p,
                bias_iters=bias_iters,
                shots=shots,
                alpha=alpha,
                p_small=p_small,
                bias_iters_small=bias_iters_small,
                shots_small=shots_small,
            )
            stage_records.append(
                {
                    "label": stage_label,
                    "sizes": sizes,
                    "pool_stats": pool_stats,
                    "result": stage_result,
                }
            )
            final_result = stage_result

            if verbose:
                print_stage_summary(stage_label, pool_stats, stage_result)

            if not final_stage:
                promising_data = extract_promising_neighborhoods(
                    current_pool,
                    stage_result["bf_result"],
                )
                if verbose:
                    print_promising_summary(promising_data)

        route_pool = current_pool

    return {
        "mode": mode,
        "route_pool": route_pool,
        "D": D,
        "stage_records": stage_records,
        "routes": final_result["routes"],
        "cost": final_result["cost"],
        "source": final_result["source"],
        "Q": final_result["Q"],
        "constant": final_result["constant"],
        "A": final_result["A"],
        "B": final_result["B"],
        "bf_result": final_result["bf_result"],
        "bbb_result": final_result["bbb_result"],
    }


single_run_result = run_single_instance_solver(
    nodes=nodes,
    Nv=Nv,
    C=C,
    route_pool_config=route_pool_config,
    use_bbb_refinement=use_bbb_refinement,
    verbose=True,
)

route_pool = single_run_result["route_pool"]
D = single_run_result["D"]
Q = single_run_result["Q"]
constant = single_run_result["constant"]
A = single_run_result["A"]
B = single_run_result["B"]
bf_result = single_run_result["bf_result"]
bbb_result = single_run_result["bbb_result"]
routes = single_run_result["routes"]
total_distance = single_run_result["cost"]
candidate_source = single_run_result["source"]

print(f"\n{'=' * 56}")
print(f"FINAL SOLUTION ({candidate_source})")
print(f"{'=' * 56}")

if routes is None:
    print("No feasible solution was decoded.")
else:
    for i, route in enumerate(routes):
        rd = route_distance(route, D)
        print(f"  r{i + 1}: {', '.join(map(str, route))}  |  Distance: {rd:.2f}")
    print(f"\nTotal Distance: {total_distance:.4f}")


In [ ]:
# -- Feasibility Check --

print("=" * 50)
print("FEASIBILITY CHECK")
print("=" * 50)

all_customers = set(range(1, len(nodes)))
visited = []
feasible = True

in_degree = {i: 0 for i in range(len(nodes))}
out_degree = {i: 0 for i in range(len(nodes))}

if routes is None:
    feasible = False
    print("  X No decoded routes available")
else:
    for i, route in enumerate(routes):
        if route[0] != 0 or route[-1] != 0:
            print(f"  X Route {i + 1} does not start/end at depot: {route}")
            feasible = False
        else:
            print(f"  OK Route {i + 1} starts and ends at depot")

        customers_in_route = [customer for customer in route if customer != 0]
        if len(customers_in_route) > C:
            print(f"  X Route {i + 1} exceeds capacity: {len(customers_in_route)} > {C}")
            feasible = False
        else:
            print(f"  OK Route {i + 1} capacity OK ({len(customers_in_route)}/{C})")

        if len(customers_in_route) != len(set(customers_in_route)):
            print(f"  X Route {i + 1} has repeated customers: {route}")
            feasible = False
        else:
            print(f"  OK Route {i + 1} has no internal repeats")

        for k in range(len(route) - 1):
            out_degree[route[k]] += 1
            in_degree[route[k + 1]] += 1

        visited.extend(customers_in_route)

    visited_set = set(visited)
    missing = all_customers - visited_set
    duplicates = [customer for customer in visited if visited.count(customer) > 1]

    if missing:
        print(f"  X Missing customers: {missing}")
        feasible = False
    else:
        print(f"  OK All {len(all_customers)} customers visited")

    if duplicates:
        print(f"  X Duplicate visits: {set(duplicates)}")
        feasible = False
    else:
        print("  OK No duplicate visits")

    if len(routes) != Nv:
        print(f"  X Vehicle-count mismatch: {len(routes)} != {Nv}")
        feasible = False
    else:
        print(f"  OK Vehicles used exactly match Nv={Nv}")

    print(f"\n{'-' * 50}")
    print("FLOW CONSERVATION")
    print(f"{'-' * 50}")

    depot_out_ok = out_degree[0] == len(routes)
    depot_in_ok = in_degree[0] == len(routes)

    if depot_out_ok and depot_in_ok:
        print(f"  OK Depot flow balanced: {out_degree[0]} out, {in_degree[0]} in")
    else:
        print(f"  X Depot flow imbalanced: {out_degree[0]} out, {in_degree[0]} in")
        feasible = False

    flow_violations = []
    for customer in all_customers:
        if in_degree[customer] != 1 or out_degree[customer] != 1:
            flow_violations.append((customer, in_degree[customer], out_degree[customer]))

    if not flow_violations:
        print("  OK All customers have in-degree = out-degree = 1")
    else:
        for customer, indeg, outdeg in flow_violations:
            print(f"  X Customer {customer}: in-degree={indeg}, out-degree={outdeg}")
        feasible = False

    print(f"\n{'-' * 50}")
    print("DISTANCE VERIFICATION")
    print(f"{'-' * 50}")

    recomputed = 0.0
    for i, route in enumerate(routes):
        rd = route_distance(route, D)
        recomputed += rd
        print(f"  Route {i + 1}: {' -> '.join(map(str, route))}  dist={rd:.2f}")

    if abs(recomputed - total_distance) < 1e-6:
        print(f"  OK Total distance verified: {recomputed:.4f}")
    else:
        print(
            f"  X Distance mismatch: reported={total_distance:.4f}, "
            f"recomputed={recomputed:.4f}"
        )
        feasible = False

print(f"\n{'=' * 50}")
print("SOLUTION FEASIBLE" if feasible else "SOLUTION INFEASIBLE")
print(f"{'=' * 50}")


In [ ]:
# -- Optional: Run All Instances with the New Solver --

summary_rows = []

for inst_id, instance_cfg in instances.items():
    Nv_inst = instance_cfg["Nv"]
    C_inst = instance_cfg["C"]
    nodes_inst = instance_cfg["nodes"]
    n_cust = len(nodes_inst) - 1

    print(f"\n{'#' * 60}")
    print(f"INSTANCE {inst_id}: {n_cust} customers, {Nv_inst} vehicles, capacity {C_inst}")
    print(f"{'#' * 60}")

    inst_result = run_single_instance_solver(
        nodes=nodes_inst,
        Nv=Nv_inst,
        C=C_inst,
        route_pool_config=route_pool_config,
        use_bbb_refinement=use_bbb_refinement,
        verbose=False,
    )

    best_routes = inst_result["routes"]
    best_cost = inst_result["cost"]
    source = inst_result["source"]
    bf_inst = inst_result["bf_result"]
    route_pool_inst = inst_result["route_pool"]
    stage_count = len(inst_result["stage_records"])

    final_stage_stats = inst_result["stage_records"][-1]["pool_stats"]
    print(
        f"  mode={inst_result['mode']}, stages={stage_count}, "
        f"route_vars={len(route_pool_inst)}, feasible_rate={bf_inst['feasible_rate']:.4f}"
    )
    print(
        "  pool stats: "
        f"before={final_stage_stats['total_candidate_subsets_before_pruning']}, "
        f"neighbor={final_stage_stats['after_neighbor_restriction']}, "
        f"geom={final_stage_stats['after_geometric_pruning']}, "
        f"cap={final_stage_stats['after_guarded_cap']}"
    )

    print(f"  Selected source: {source}")
    if best_routes is not None:
        for i, route in enumerate(best_routes):
            print(
                f"  r{i + 1}: {' -> '.join(map(str, route))}  |  "
                f"dist={route_distance(route, inst_result['D']):.2f}"
            )
        print(f"  Total distance: {best_cost:.4f}")
    else:
        print("  No feasible solution decoded")

    summary_rows.append(
        (
            inst_id,
            n_cust,
            Nv_inst,
            C_inst,
            len(route_pool_inst),
            best_cost,
            bf_inst["max_qubits"],
            bf_inst["total_gates"],
            bf_inst["total_time"],
            source,
            best_routes is not None,
        )
    )

print(f"\n\n{'=' * 118}")
print(f"{'SUMMARY':^118}")
print(f"{'=' * 118}")
print(
    f"{'Inst':>5} {'Cust':>5} {'Veh':>4} {'Cap':>4} {'Vars':>6} "
    f"{'Distance':>12} {'Qubits':>7} {'Gates':>8} {'Time(s)':>9} {'Source':>20} {'Status':>12}"
)
print(f"{'-' * 118}")
for inst_id, nc, nv, cap, vars_count, dist, qb, gt, tm, src, ok in summary_rows:
    tag = "FEASIBLE" if ok else "INFEASIBLE"
    print(
        f"{inst_id:>5} {nc:>5} {nv:>4} {cap:>4} {vars_count:>6} "
        f"{dist:>12.4f} {qb:>7} {gt:>8} {tm:>9.2f} {src:>20} {tag:>12}"
    )
print(f"{'=' * 118}")
